# How to Run Llama 3 Locally: A Complete Guide

https://www.datacamp.com/tutorial/run-llama-3-locally?utm_source=google&utm_medium=paid_search&utm_campaignid=19589720821&utm_adgroupid=157098104375&utm_device=c&utm_keyword=&utm_matchtype=&utm_network=g&utm_adpostion=&utm_creative=720362650267&utm_targetid=dsa-2264919291989&utm_loc_interest_ms=&utm_loc_physical_ms=9186691&utm_content=&utm_campaign=230119_1-sea~dsa~tofu_2-b2c_3-row-p1_4-prc_5-na_6-na_7-le_8-pdsh-go_9-nb-e_10-na_11-na-bfcm24&gad_source=1&gclid=EAIaIQobChMIhN-x15WQigMVz5eDBx2cpx_6EAAYASAAEgLLvfD_BwE

## Loading the documents

In [1]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader("../data", glob="**/*.txt")
books = loader.load()
len(books)

3

## Splitting the text

Documents are cut into chunks of 500 characters using `RecursiveCharacterTextSplitter`, which tries to split at natural boundaries (paragraphs → sentences → words) before falling back to raw character cuts.

**Why this is necessary:**
- **Embedding quality** — each chunk is embedded as a vector and retrieved by similarity. Embedding a whole document averages out its meaning; small, focused chunks produce vectors that represent a single idea, so retrieval actually finds the right passage.
- **Context window limits** — LLMs can only receive a limited amount of text per call. Only the most relevant chunks are sent to the model, not the entire document set.

**Note on `chunk_overlap`:** currently set to `0`, meaning chunks share no content. If a key sentence falls on a chunk boundary, neither chunk contains the full thought. A small overlap (50–100 characters) is usually safer.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
all_splits = text_splitter.split_documents(books)

## Ollama embeddings and Chroma vector store

In [3]:
from langchain_chroma import Chroma
from langchain_community.embeddings import OllamaEmbeddings

vectorstore = Chroma.from_documents(
    documents=all_splits,
    embedding=OllamaEmbeddings(model="llama3", show_progress=True),
    persist_directory="../chroma_db",
)

/var/folders/xt/bk2xvl154816mgqr2nsycpyw0000gn/T/ipykernel_24123/435689994.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding=OllamaEmbeddings(model="llama3", show_progress=True),
OllamaEmbeddings: 100%|██████████| 3/3 [00:09<00:00,  3.21s/it]


In [4]:
question = "Was für Kupplungen hat Lieferant X?"
docs = vectorstore.similarity_search(question)
docs

OllamaEmbeddings: 100%|██████████| 1/1 [00:01<00:00,  1.71s/it]
Number of requested results 4 is greater than number of elements in index 3, updating n_results = 3


[Document(metadata={'source': '../data/text_3.txt'}, page_content='2024/12/05 10:00\n\nDer Lieferant Z vertreibt die Kupplung B.'),
 Document(metadata={'source': '../data/text_1.txt'}, page_content='2024/12/05 08:30\n\nDer Lieferant X vertreibt die Kupplung B.'),
 Document(metadata={'source': '../data/text_2.txt'}, page_content='2024/12/05 09:30\n\nDer Lieferant X vertreibt die Kupplung C.')]

## Building Langchain chains for Q&A retrieval system

In [5]:
from langchain import hub
from langchain_community.llms import Ollama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = Ollama(model="llama3")

retriever = vectorstore.as_retriever()


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_prompt = hub.pull("rlm/rag-prompt")
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

/var/folders/xt/bk2xvl154816mgqr2nsycpyw0000gn/T/ipykernel_24123/2695149408.py:6: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3")
/Users/davidfischer/miniconda3/envs/chatbot/lib/python3.12/site-packages/langsmith/client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


## Testing the Q&A retrieval chain

In [6]:
question = "Welche Kupplungen bietet Lieferant X an?"
qa_chain.invoke(question)

OllamaEmbeddings: 100%|██████████| 1/1 [00:01<00:00,  1.97s/it]
Number of requested results 4 is greater than number of elements in index 3, updating n_results = 3


'According to the given context, Lieferant X offers Kupplung B and Kupplung C.'